Сайт, на котором я запускала SQL: https://dbfiddle.dev/postgres?checkpoint=1789057563

## Стартовый датасет для первой задачи

```sql
DROP TABLE IF EXISTS campaigns;

CREATE TABLE campaigns (
    user_id INT,
    campaign_id INT,
    channel TEXT,
    sent_at DATE,
    clicked INT,
    purchased INT,
    revenue NUMERIC
);

INSERT INTO campaigns (user_id, campaign_id, channel, sent_at, clicked, purchased, revenue) VALUES
(1, 101, 'push',  '2026-09-01', 1, 1, 1200),
(2, 101, 'push',  '2026-09-01', 0, 0, 0),
(3, 101, 'push',  '2026-09-01', 1, 0, 0),
(4, 102, 'email', '2026-09-02', 1, 1, 2500),
(5, 102, 'email', '2026-09-02', 0, 0, 0),
(6, 102, 'email', '2026-09-02', 1, 1, 1800),
(7, 103, 'sms',   '2026-09-03', 0, 0, 0),
(8, 103, 'sms',   '2026-09-03', 1, 0, 0),
(9, 103, 'sms',   '2026-09-03', 1, 1, 900);
```

## Задача 1: Метрики По Кампаниям

Таблица campaigns уже создана.

Нужно по каждой campaign_id посчитать:

1. количество отправок;
2. количество кликов;
3. количество покупок;
4. CTR = клики / отправки;
5. conversion rate = покупки / отправки;
6. суммарную выручку;
7. среднюю выручку на отправку.
   
Отсортировать результат по campaign_id.

Ожидаемые колонки:

```sql
campaign_id
sends
clicks
purchases
ctr
conversion_rate
total_revenue
revenue_per_send
```

### Решение:

```sql
SELECT 
    campaign_id,
    COUNT(*) AS sends,
    SUM(clicked) AS clicks,
    SUM(purchased) AS purchases,
    SUM(clicked) * 1.0 / COUNT(*) AS ctr,
    SUM(purchased) * 1.0 / COUNT(*) AS conversion_rate,
    SUM(revenue) AS total_revenue,
    SUM(revenue) * 1.0 / COUNT(*) AS revenue_per_send
FROM campaigns
GROUP BY campaign_id
ORDER BY campaign_id;
```

## Стартовый датасет для второй задачи

```sql
DROP TABLE IF EXISTS purchases;
DROP TABLE IF EXISTS clients;

CREATE TABLE clients (
    client_id INT,
    client_name TEXT,
    segment TEXT
);

CREATE TABLE purchases (
    purchase_id INT,
    client_id INT,
    purchase_dt DATE,
    status TEXT,
    amount NUMERIC
);

INSERT INTO clients (client_id, client_name, segment) VALUES
(1, 'Anna', 'premium'),
(2, 'Boris', 'mass'),
(3, 'Daria', 'mass'),
(4, 'Elena', 'premium'),
(5, 'Ivan', 'mass');

INSERT INTO purchases (purchase_id, client_id, purchase_dt, status, amount) VALUES
(1001, 1, '2026-09-01', 'paid', 1200),
(1002, 1, '2026-09-03', 'cancelled', 500),
(1003, 2, '2026-09-02', 'paid', 800),
(1004, 2, '2026-09-05', 'paid', 1500),
(1005, 4, '2026-09-04', 'cancelled', 700);
```

## Задача 2: Вывести всех клиентов и по каждому посчитать:

1. paid_purchases — количество оплаченных покупок;
2. paid_amount — сумму оплаченных покупок;
3. has_paid_purchase — флаг 1, если у клиента была хотя бы одна оплаченная покупка, иначе 0.
   
Ожидаемые колонки:

```sql
client_id,
client_name,
segment,
paid_purchases,
paid_amount,
has_paid_purchase
```

**Важно**: клиенты без покупок тоже должны остаться в результате.

Отсортируй по client_id.

### Решение:

```sql
SELECT 
    c.client_id,
    c.client_name,
    c.segment,
    COUNT(p.purchase_id) AS paid_purchases,
    COALESCE(SUM(p.amount), 0) AS paid_amount,
    CASE 
        WHEN COUNT(p.purchase_id) > 0
        THEN 1
        ELSE 0
    END AS has_paid_purchase
FROM clients AS c
LEFT JOIN purchases AS p 
    ON c.client_id = p.client_id AND p.status = 'paid'
GROUP BY c.client_id, c.client_name, c.segment
ORDER BY c.client_id;
```

## Стартовый датасет для третьей задачи

```sql
DROP TABLE IF EXISTS payments;

CREATE TABLE payments (
    payment_id INT,
    client_id INT,
    payment_dt DATE,
    amount NUMERIC
);

INSERT INTO payments (payment_id, client_id, payment_dt, amount) VALUES
(1, 101, '2026-09-01', 500),
(2, 101, '2026-09-03', 700),
(3, 101, '2026-09-05', 300),
(4, 102, '2026-09-01', 1000),
(5, 102, '2026-09-04', 200),
(6, 103, '2026-09-02', 400),
(7, 103, '2026-09-06', 600);
```

## Задача 3: На оконные функции: ROW_NUMBER, SUM OVER, LAG.

Для каждой оплаты вывести:

```sql
payment_id,
client_id,
payment_dt,
amount,
payment_number,
client_total_amount,
running_amount,
previous_amount
```
    
Где:
    
- payment_number — номер оплаты клиента по дате;
- client_total_amount — общая сумма оплат клиента;
- running_amount — накопительная сумма оплат клиента на дату этой оплаты;
- previous_amount — сумма предыдущей оплаты этого же клиента.
    
Отсортировать результат по client_id, payment_dt.

### Решение:

```sql
SELECT 
    payment_id,
    client_id, 
    payment_dt,
    amount,
    ROW_NUMBER() 
        OVER (
            PARTITION BY client_id
            ORDER BY payment_dt, payment_id) AS payment_number,
    SUM(amount) 
        OVER(
            PARTITION BY client_id) AS client_total_amount,
    SUM(amount) 
        OVER(
            PARTITION BY client_id
            ORDER BY payment_dt, payment_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_amount,
    LAG(amount)
        OVER(
            PARTITION BY client_id
            ORDER BY payment_dt, payment_id) AS previous_amount
FROM payments p 
ORDER BY client_id, payment_dt;
```

## Стартовый датасет для четвертой задачи

```sql
DROP TABLE IF EXISTS client_status_log;

CREATE TABLE client_status_log (
    log_id INT,
    client_id INT,
    status_dt DATE,
    status TEXT
);

INSERT INTO client_status_log (log_id, client_id, status_dt, status) VALUES
(1, 101, '2026-09-01', 'new'),
(2, 101, '2026-09-03', 'active'),
(3, 101, '2026-09-05', 'inactive'),
(4, 102, '2026-09-01', 'new'),
(5, 102, '2026-09-04', 'active'),
(6, 103, '2026-09-02', 'new'),
(7, 103, '2026-09-02', 'active'),
(8, 104, '2026-09-06', 'new');
```

## Задача 4: Нужно вывести по каждому клиенту его последний статус.

Ожидаемые колонки:

```sql
client_id,
last_status_dt,
last_status
```
    
**Важно**: у клиента может быть несколько статусов в один день, поэтому порядок нужно сделать однозначным. Тут для этого есть log_id.

Отсортируй результат по client_id.

### Решение:

```sql
WITH last_status_of_client AS (
    SELECT 
        log_id,
        client_id,
        status_dt,
        status,
        ROW_NUMBER() 
            OVER(
                PARTITION BY client_id
                ORDER BY status_dt DESC, log_id DESC) AS row_num
    FROM client_status_log
)
SELECT 
    client_id,
    status_dt AS last_status_dt,
    status AS last_status 
FROM last_status_of_client
WHERE row_num = 1
ORDER BY client_id;
```

## Стартовый датасет для пятой задачи

```sql
DROP TABLE IF EXISTS campaign_events;

CREATE TABLE campaign_events (
    event_id INT,
    user_id INT,
    campaign_id INT,
    event_type TEXT,
    event_dt TIMESTAMP
);

INSERT INTO campaign_events (event_id, user_id, campaign_id, event_type, event_dt) VALUES
(1, 1, 101, 'sent',     '2026-09-01 10:00:00'),
(2, 1, 101, 'click',    '2026-09-01 10:05:00'),
(3, 1, 101, 'purchase', '2026-09-01 10:20:00'),

(4, 2, 101, 'sent',     '2026-09-01 11:00:00'),

(5, 3, 101, 'sent',     '2026-09-01 12:00:00'),
(6, 3, 101, 'click',    '2026-09-01 12:10:00'),
(7, 3, 101, 'click',    '2026-09-01 12:15:00'),

(8, 4, 102, 'sent',     '2026-09-02 09:00:00'),
(9, 4, 102, 'click',    '2026-09-02 09:10:00'),
(10, 4, 102, 'purchase','2026-09-02 09:30:00'),

(11, 5, 102, 'sent',    '2026-09-02 10:00:00'),
(12, 5, 102, 'purchase','2026-09-02 10:40:00'),

(13, 6, 102, 'sent',    '2026-09-02 11:00:00'),

(14, 7, 103, 'sent',    '2026-09-03 10:00:00'),
(15, 8, 103, 'sent',    '2026-09-03 10:05:00'),
(16, 8, 103, 'click',   '2026-09-03 10:20:00');
```

## Задача 5: воронка по кампаниям

По каждой campaign_id посчитать:

```sql
campaign_id,
sent_users,
clicked_users,
purchased_users,
ctr,
conversion_rate,
click_to_purchase_rate
```
    
Где:
    
- sent_users — сколько уникальных пользователей получили коммуникацию;
- clicked_users — сколько уникальных пользователей кликнули;
- purchased_users — сколько уникальных пользователей купили;
- ctr = clicked_users / sent_users;
- conversion_rate = purchased_users / sent_users;
- click_to_purchase_rate = purchased_users / clicked_users.
    
**Важно**: один пользователь может иметь несколько событий одного типа, поэтому считаем именно уникальных пользователей, а не строки.

Отсортируй по campaign_id.

### Решение 1: С помощью CTE

```sql
WITH 
count_sent AS 
(
    SELECT campaign_id,
        COUNT(DISTINCT user_id) AS count_s
    FROM campaign_events 
    WHERE event_type = 'sent'
    GROUP BY campaign_id
),
count_clicked AS 
(
    SELECT campaign_id,
        COUNT(DISTINCT user_id) AS count_cl
    FROM campaign_events 
    WHERE event_type = 'click'
    GROUP BY campaign_id
),
count_purchased AS 
(
    SELECT campaign_id,
        COUNT(DISTINCT user_id) AS count_p
    FROM campaign_events 
    WHERE event_type = 'purchase'
    GROUP BY campaign_id
)
SELECT 
    count_sent.campaign_id,
    count_sent.count_s AS sent_users,
    COALESCE(count_clicked.count_cl, 0) AS clicked_users,
    COALESCE(count_purchased.count_p, 0) AS purchased_users,
    COALESCE(count_clicked.count_cl, 0) * 1.0 / NULLIF(count_sent.count_s, 0) AS ctr,
    COALESCE(count_purchased.count_p, 0) * 1.0 / NULLIF(count_sent.count_s, 0) AS conversion_rate,
    COALESCE(count_purchased.count_p, 0) * 1.0 / NULLIF(count_clicked.count_cl, 0) AS click_to_purchase_rate
FROM count_sent      
LEFT JOIN count_clicked   ON count_sent.campaign_id = count_clicked.campaign_id
LEFT JOIN count_purchased ON count_sent.campaign_id = count_purchased.campaign_id
ORDER BY count_sent.campaign_id;
```

### Решение 2: С помощью CTE + условной агрегации

```sql
WITH 
counts_CTE AS 
(
    SELECT 
        campaign_id,
        COUNT(DISTINCT 
            CASE 
                WHEN event_type = 'sent'
                THEN user_id
            END) AS sent_users,
        COUNT(DISTINCT 
            CASE 
                WHEN event_type = 'click'
                THEN user_id
            END) AS clicked_users,
        COUNT(DISTINCT 
            CASE 
                WHEN event_type = 'purchase'
                THEN user_id
            END) AS purchased_users
    FROM campaign_events
    GROUP BY campaign_id
)
SELECT 
    campaign_id,
    sent_users,
    clicked_users,
    purchased_users,
    clicked_users * 1.0 / NULLIF(sent_users, 0) AS ctr,
    purchased_users * 1.0 / NULLIF(sent_users, 0) AS conversion_rate,
    purchased_users * 1.0 / NULLIF(clicked_users, 0) AS click_to_purchase_rate
FROM counts_CTE
ORDER BY campaign_id;
```